# 00 - Setup landing zone

One-time (and re-runnable) setup for the sdp-meta retail demo:

1. Creates the five `sdp_meta_*` schemas and the two volumes (`sdp_meta_landing.files`, `sdp_meta_control.config`).
2. Copies the **baseline** `retail-org` files from `samples.databricks.datasets` into the landing volume, one folder per group / table.
3. Verifies row counts in the landing zone against the samples source.

Landing layout:

```
/Volumes/<catalog>/sdp_meta_landing/files/
  retail_reference/customers/          CSV  (+ operation column for SCD2 / CDC)
  retail_reference/products/           CSV  (';' delimited)
  retail_reference/loyalty_segments/   CSV
  retail_reference/promotions/         CSV
  retail_reference/active_promotions/  Parquet
  retail_reference/suppliers/          CSV  (onboarded live in the demo)
  retail_sales/sales_orders/           JSON (nested)
  retail_sales/sales_stream/           JSON (first 15 of 45 files; rest fed by 01_simulate_increments)
```

In [0]:
dbutils.widgets.text("catalog", "mg_aws_serverless_workspace_catalog", "Catalog")
dbutils.widgets.text("prefix", "sdp_meta", "Schema prefix")
dbutils.widgets.text("sales_stream_baseline_files", "15", "sales_stream files in baseline")

CATALOG = dbutils.widgets.get("catalog")
PREFIX = dbutils.widgets.get("prefix")
STREAM_BASELINE = int(dbutils.widgets.get("sales_stream_baseline_files"))

SCHEMAS = {layer: f"{PREFIX}_{layer}" for layer in ["landing", "control", "bronze", "silver", "gold"]}
SRC = "/Volumes/samples/databricks/datasets/retail-org"
LANDING = f"/Volumes/{CATALOG}/{SCHEMAS['landing']}/files"
REF = f"{LANDING}/retail_reference"
SALES = f"{LANDING}/retail_sales"
print(SCHEMAS, LANDING, sep="\n")

In [0]:
# IDENTIFIER() + named parameters avoids string-built SQL identifiers.
for layer, schema in SCHEMAS.items():
    spark.sql(
        "CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:fqn) COMMENT :c",
        args={"fqn": f"{CATALOG}.{schema}", "c": f"sdp-meta retail demo - {layer} layer"},
    )

spark.sql("CREATE VOLUME IF NOT EXISTS IDENTIFIER(:v) COMMENT 'Landing zone for retail-org source files'",
          args={"v": f"{CATALOG}.{SCHEMAS['landing']}.files"})
spark.sql("CREATE VOLUME IF NOT EXISTS IDENTIFIER(:v) COMMENT 'sdp-meta onboarding, DQE and silver transformation config (staged by the bundle)'",
          args={"v": f"{CATALOG}.{SCHEMAS['control']}.config"})

display(spark.sql("SHOW SCHEMAS IN IDENTIFIER(:c) LIKE :p", args={"c": CATALOG, "p": f"{PREFIX}_*"}))

In [0]:
# Files copied unchanged; one folder per table so each sdp-meta source_path maps to exactly one table.
AS_IS = {
    "products": "products/products.csv",
    "loyalty_segments": "loyalty_segments/loyalty_segment.csv",
    "promotions": "promotions/promotions.csv",
    "active_promotions": "active_promotions/active_promotions.parquet",
    "suppliers": "suppliers/suppliers.csv",
}
for table, rel in AS_IS.items():
    fname = rel.split("/")[-1]
    dbutils.fs.cp(f"{SRC}/{rel}", f"{REF}/{table}/{fname}")
    print(f"copied {rel} -> retail_reference/{table}/{fname}")

In [0]:
# customers.csv already holds history (multiple versions per customer_id, ordered by valid_from).
# Add CDC columns so the same folder can later receive UPDATE / DELETE change files (batch 2):
#   operation     - INSERT for the baseline
#   operation_ts  - sequence column for SCD2 = valid_from (epoch seconds)
# Small file (~29K rows) -> pandas keeps a single, cleanly named CSV in the volume.
cust = spark.read.option("header", True).csv(f"{SRC}/customers")
cust_pdf = cust.toPandas()
cust_pdf["operation"] = "INSERT"
cust_pdf["operation_ts"] = cust_pdf["valid_from"].astype("int64")

dbutils.fs.mkdirs(f"{REF}/customers")
cust_pdf.to_csv(f"{REF}/customers/customers_000_initial.csv", index=False)
print(f"wrote {len(cust_pdf):,} rows -> retail_reference/customers/customers_000_initial.csv")

In [0]:
# sales_orders: single JSON part file -> renamed copy (Spark commit markers are skipped)
so_part = [f for f in dbutils.fs.ls(f"{SRC}/sales_orders") if f.name.endswith(".json")]
assert len(so_part) == 1, so_part
dbutils.fs.cp(so_part[0].path, f"{SALES}/sales_orders/sales_orders_000_initial.json")

# sales_stream: copy only the first N files (sorted by name = time order); the rest are
# dropped in later by 01_simulate_increments to demonstrate incremental Auto Loader.
stream_files = sorted(dbutils.fs.ls(f"{SRC}/sales_stream/sales_stream.json"), key=lambda f: f.name)
for f in stream_files[:STREAM_BASELINE]:
    dbutils.fs.cp(f.path, f"{SALES}/sales_stream/{f.name}")
print(f"sales_stream: copied {STREAM_BASELINE} of {len(stream_files)} files")

In [0]:
# Independent check: re-read both sides and compare row counts.
checks = [
    ("customers", spark.read.option("header", True).csv(f"{SRC}/customers"),
                  spark.read.option("header", True).csv(f"{REF}/customers")),
    ("products", spark.read.text(f"{SRC}/products"), spark.read.text(f"{REF}/products")),
    ("loyalty_segments", spark.read.option("header", True).csv(f"{SRC}/loyalty_segments"),
                         spark.read.option("header", True).csv(f"{REF}/loyalty_segments")),
    # promotions has a quoted value spanning a line break -> multiLine/escape (3 real rows)
    ("promotions", spark.read.options(header=True, multiLine=True, escape='"').csv(f"{SRC}/promotions"),
                   spark.read.options(header=True, multiLine=True, escape='"').csv(f"{REF}/promotions")),
    ("active_promotions", spark.read.parquet(f"{SRC}/active_promotions"), spark.read.parquet(f"{REF}/active_promotions")),
    ("suppliers", spark.read.option("header", True).csv(f"{SRC}/suppliers"),
                  spark.read.option("header", True).csv(f"{REF}/suppliers")),
    ("sales_orders", spark.read.json(f"{SRC}/sales_orders"), spark.read.json(f"{SALES}/sales_orders")),
]
rows = [(name, s.count(), t.count()) for name, s, t in checks]
expected_stream = sum(spark.read.json(f.path).count() for f in stream_files[:STREAM_BASELINE])
rows.append(("sales_stream (baseline files)", expected_stream, spark.read.json(f"{SALES}/sales_stream").count()))

result = spark.createDataFrame(rows, "table STRING, source_rows LONG, landing_rows LONG") \
              .selectExpr("*", "source_rows = landing_rows AS match")
display(result)
assert all(r.match for r in result.collect()), "Row count mismatch between samples and landing"